# 05 — Voice loop (Siri-style)

`speech → MLX Whisper (small) → R2 → macOS say`

Part 1 needs **no microphone**: phrases are synthesised with `say`, transcribed,
and routed. Part 2 is a push-to-talk widget (grant microphone access to the app
running Jupyter, e.g. Terminal / VS Code / Cursor).

In [1]:
# --- setup: make the repo importable and quiet logs ---
import os, sys, logging
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_OFFLINE", "1")  # models are already cached → fully local
logging.basicConfig(level=logging.WARNING)
print("repo:", ROOT)

repo: /Users/siavash/Projects/articles/narrow-ai-with-kv


In [2]:
from narrow_ai.r2 import R2
r2 = R2(workspace=ROOT)
SPEAK = False   # set True to hear R2's replies

## Part 1 · synthetic speech round-trip

In [3]:
import tempfile, time
import pandas as pd
from IPython.display import Audio, display
from narrow_ai.speech.transcription import synthesize_to_wav

phrases = ["Hey R2, what's my git status?", "Find all the Python files.",
           "Show me the read me file.", "Open Xcode.", "Order me a pizza."]
rows, tmp = [], Path(tempfile.mkdtemp())
for i, p in enumerate(phrases):
    wav = synthesize_to_wav(p, tmp / f"p{i}.wav")
    t0 = time.perf_counter(); heard = r2.transcribe(wav); stt = (time.perf_counter() - t0) * 1000
    resp = r2.ask(heard); r2.cancel()
    rows.append({"said": p, "heard": heard, "stt_ms": round(stt), "intent": resp.decision.intent.value,
                 "R2 says": resp.spoken_text(), "think_ms": resp.metrics.get("think_ms")})
display(Audio(str(tmp / "p0.wav")))
pd.DataFrame(rows)

,said,heard,stt_ms,intent,R2 says,think_ms
0,"Hey R2, what's my git status?","Hey R2, what's my git status?",2283,git_status,You're on main with 23 changed files.,377.3
1,Find all the Python files.,Find all the Python files.,393,search_files,I found 50 files or more matching *.py.,635.6
2,Show me the read me file.,Show me the README file.,319,read_file,"Here's README.md, 96 lines shown.",433.9
3,Open Xcode.,Open Xcode.,307,open_app,Open Xcode? Say yes or no.,487.0
4,Order me a pizza.,Order me a pizza.,364,unsupported,"Sorry, that's outside what I can do.",490.1


In [4]:
if SPEAK:
    r2.speak(rows[0]["R2 says"])

## Part 2 · push-to-talk with your microphone

In [5]:
import ipywidgets as W
talk = W.ToggleButton(description="🎙 Hold to talk (click, speak, click)", button_style="info",
                      layout=W.Layout(width="320px"))
out = W.Output()
def on_toggle(change):
    with out:
        if change["new"]:
            out.clear_output(); r2.listen_start(); print("listening…")
        else:
            try:
                heard, resp = r2.listen_stop(speak=SPEAK)
                print("heard:", heard or "(nothing)")
                if resp:
                    print("R2   :", resp.spoken_text())
                    if resp.text != resp.spoken_text():
                        print(resp.text)
            except Exception as e:
                print("voice error:", e)
talk.observe(on_toggle, "value")
display(talk, out)

ToggleButton(value=False, button_style='info', description='🎙 Hold to talk (click, speak, click)', layout=Layo…

Output()